In [137]:
import re
import os
from typing import List, Dict, Tuple
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [148]:
def parse_heading_level(line: str):
    stripped = line.strip()
    if not stripped:
        return None, None
    tokens = stripped.split()
    if not tokens:
        return None, None

    left_eq = 0
    for token in tokens:
        if token == '=':
            left_eq += 1
        else:
            break
    if left_eq == 0:
        return None, None

    right_eq = 0
    for token in reversed(tokens):
        if token == '=':
            right_eq += 1
        else:
            break

    if left_eq != right_eq or left_eq * 2 >= len(tokens):
        return None, None

    title_tokens = tokens[left_eq : -right_eq]
    title = ' '.join(title_tokens).strip()
    return (left_eq, title) if title else (None, None)


def wikitext_to_nested_json(text: str):
    lines = text.splitlines()
    result = {}
    
    current_top_title = None
    current_tree = None
    section_stack = []  # [(level, title), ...]
    buffer = []

    def flush_content():
        content = "\n".join(buffer).strip()
        if not content:
            return
        if not section_stack:
            # 引言
            current_tree["__intro__"] = content
        else:
            # 确保路径上所有节点都是 dict
            node = current_tree
            path = [item[1] for item in section_stack]
            for key in path[:-1]:
                if key not in node or not isinstance(node[key], dict):
                    node[key] = {}
                node = node[key]
            # 最后一级：直接赋值
            node[path[-1]] = content
        buffer.clear()

    for line in lines:
        level, title = parse_heading_level(line)
        if level is not None:
            # 先 flush 当前 buffer（如果有内容）
            flush_content()
            
            if level == 1:
                # 顶级条目
                if current_top_title is not None:
                    result[current_top_title] = current_tree
                current_top_title = title
                current_tree = {"__intro__": ""}
                section_stack = []
            else:
                # 章节标题
                while section_stack and section_stack[-1][0] >= level:
                    section_stack.pop()
                section_stack.append((level, title))
        else:
            if current_top_title is not None:
                buffer.append(line)
    
    # flush last
    flush_content()
    if current_top_title is not None:
        result[current_top_title] = current_tree

    return result


def flatten_nested_json_to_chunks(nested_json: dict, source_file: str = "unknown.txt"):
    """
    将嵌套的 JSON 结构（由 wikitext 解析而来）转换为带标题路径前缀的扁平 chunks。
    
    Args:
        nested_json (dict): 格式如 { "Entity": { "__intro__": "...", "Section": { "Sub": "..." } } }
        source_file (str): 源文件名，用于 metadata
    
    Returns:
        List[Dict]: 每个元素为 { "text": "...", "metadata": { "entity": ..., "section_path": ... } }
    """
    chunks = []

    def recurse(node, current_path, entity):
        if isinstance(node, str):
            if not node.strip():
                return
            # 构造完整路径显示
            display_path = " / ".join([entity] + current_path) if current_path else entity
            section_path = "/".join(current_path) if current_path else entity
            chunks.append({
                "text": f"[{display_path}]\n{node}",
                "metadata": {
                    "entity": entity,
                    "section_path": section_path,
                    "source_file": source_file
                }
            })
        elif isinstance(node, dict):
            for key, value in node.items():
                if key == "__intro__":
                    # 引言段落，路径仅为 entity
                    if isinstance(value, str) and value.strip():
                        chunks.append({
                            "text": f"[{entity}]\n{value}",
                            "metadata": {
                                "entity": entity,
                                "section_path": entity,
                                "source_file": source_file
                            }
                        })
                else:
                    recurse(value, current_path + [key], entity)

    for entity, content in nested_json.items():
        if isinstance(content, dict):
            recurse(content, [], entity)
        elif isinstance(content, str) and content.strip():
            # 极端情况：整个条目是字符串（无子节）
            chunks.append({
                "text": f"[{entity}]\n{content}",
                "metadata": {
                    "entity": entity,
                    "section_path": entity,
                    "source_file": source_file
                }
            })

    return chunks

def rechunk_flat_chunks_by_words(
    flat_chunks,
    min_words=500,
    max_words=1500,
    overlap_words=300,
    path_keep=True
):
    """
    将已有的扁平 chunks（每个含 [Entity / Path] 前缀）按 entity 合并，
    并重新分块为 500–1500 词、300 词 overlap 的新 chunks。

    Args:
        flat_chunks (list): 列表，每个元素为 {"text": "...", "metadata": {"entity": "...", "source_file": "..."}}
        min_words (int): 最小词数
        max_words (int): 最大词数
        overlap_words (int): 重叠词数

    Returns:
        list: 新的 chunks 列表，格式相同（metadata 中仍含 entity 和 source_file）
    """
    from collections import defaultdict

    # Step 1: 按 entity 分组
    entity_groups = defaultdict(list)
    for chunk in flat_chunks:
        entity = chunk["metadata"]["entity"]
        source_file = chunk["metadata"].get("source_file", "unknown.txt")
        entity_groups[entity].append((chunk["text"], source_file))

    new_chunks = []

    for entity, text_source_pairs in entity_groups.items():
        # Step 2: 拼接所有文本（保留 [Path] 前缀）
        full_text = "\n\n".join(text for text, _ in text_source_pairs)
        source_file = text_source_pairs[0][1]  # 取第一个的 source_file

        words = full_text.split()
        total = len(words)

        if total == 0:
            continue

        # Step 3: 若全文 < min_words，直接作为一块
        if total < min_words:
            new_chunks.append({
                "text": full_text,
                "metadata": {"entity": entity, "source_file": source_file}
            })
            continue

        # Step 4: 滑动窗口分块
        step = max_words - overlap_words
        if step <= 0:
            step = max_words

        start = 0
        while start < total:
            end = start + max_words
            chunk_words = words[start:end]
            chunk_text = " ".join(chunk_words).strip()

            if chunk_text:
                new_chunks.append({
                    "text": chunk_text,
                    "metadata": {"entity": entity, "source_file": source_file}
                })

            if end >= total:
                break
            start += step

        # Step 5: 确保最后一个 chunk ≥ min_words（可选）
        if new_chunks and len(new_chunks[-1]["text"].split()) < min_words and total >= min_words:
            # 从后往前取 min_words
            last_start = max(0, total - min_words)
            last_text = " ".join(words[last_start:]).strip()
            if last_text:
                new_chunks[-1] = {
                    "text": last_text,
                    "metadata": {"entity": entity, "source_file": source_file}
                }

    return new_chunks

In [ ]:
input_file =  "../data/wikitext-103/wiki.train.txt"  # 替换为你的文件路径
output_file = "corpus.json"

with open(input_file, 'r', encoding='utf-8') as f:
    raw = f.read()

nested = wikitext_to_nested_json(raw)

# print(f"✅ 成功生成嵌套 JSON: {output_file}")
print(f"共解析 {len(nested)} 个顶级条目")

共解析 28800 个顶级条目


In [163]:
flat_chunks = flatten_nested_json_to_chunks(nested, source_file="wikitext_test")

In [164]:
len(flat_chunks)

244272

In [165]:
chunks_merge = rechunk_flat_chunks_by_words(
    flat_chunks,
    min_words=500,
    max_words=1500,
    overlap_words=300
)

In [166]:
chunks_merge[1]

{'text': 'Raven , attached to the ideal of Darcsen independence proposed by their leader , Dahau . At the same time , elements within Gallian Army Command move to erase the Nameless in order to protect their own interests . Hounded by both allies and enemies , and combined with the presence of a traitor within their ranks , the 422nd desperately move to keep themselves alive while at the same time fight to help the Gallian war effort . This continues until the Nameless \'s commanding officer , Ramsey Crowe , who had been kept under house arrest , is escorted to the capital city of <unk> in order to present evidence exonerating the weary soldiers and expose the real traitor , the Gallian General that had accused Kurt of Treason . Partly due to these events , and partly due to the major losses in manpower Gallia suffers towards the end of the war with the Empire , the Nameless are offered a formal position as a squad in the Gallian Army rather than serve as an anonymous shadow force . Th

In [167]:
chunks_merge[2]

{'text': 'game . PlayStation Official Magazine - UK praised the story \'s blurring of Gallia \'s moral standing , art style , and most points about its gameplay , positively noting the latter for both its continued quality and the tweaks to balance and content . Its one major criticism were multiple difficulty spikes , something that had affected the previous games . Heath Hindman of gaming website PlayStation Lifestyle praised the addition of non @-@ linear elements and improvements or removal of mechanics from Valkyria Chronicles II in addition to praising the returning gameplay style of previous games . He also positively noted the story \'s serious tone . Points criticized in the review were recycled elements , awkward cutscenes that seemed to include all characters in a scene for no good reason , pacing issues , and occasional problems with the game \'s AI . In a preview of the TGS demo , Ryan Geddes of IGN was left excited as to where the game would go after completing the demo ,

In [168]:
len(chunks_merge)

85342

In [172]:
output_file = "chunks_with_id_title.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    for i, chunk in enumerate(chunks_merge):
        new_chunk = {
            "id": i,
            "title": chunk["metadata"]["entity"],
            "text": chunk["text"],
            "metadata": chunk["metadata"]
        }
        f.write(json.dumps(new_chunk, ensure_ascii=False) + '\n')